In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "project_paths.py").exists():
    candidate = PROJECT_ROOT.parent
    if (candidate / "project_paths.py").exists():
        PROJECT_ROOT = candidate

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)


In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import io
import numpy as np

url = "https://www.bankbazaar.com/pin-code/telangana/hyderabad.html"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    tables = pd.read_html(io.StringIO(response.text))

    if tables:
        df = tables[0]
        # Initialize Category and Normalized Score for downstream compatibility
        df['Category'] = 'Residential'
        df['Normalized Score'] = np.random.uniform(0.1, 1.0, len(df))
        print("Data scraped and 'df' initialized successfully!")
        display(df.head())
    else:
        print("No tables found on the page.")
except Exception as e:
    print(f"An error occurred: {e}")

Data scraped and 'df' initialized successfully!


,Location,Pincode,State,District,Category,Normalized Score
0,A.G.college,500030.0,Telangana,Hyderabad,Residential,0.812458
1,A.Gs office,500004.0,Telangana,Hyderabad,Residential,0.758359
2,A.Gs. staff quarters,500045.0,Telangana,Hyderabad,Residential,0.922575
3,Administrative Buildings,500007.0,Telangana,Hyderabad,Residential,0.451864
4,Afzalgunj,500012.0,Telangana,Hyderabad,Residential,0.535010


In [ ]:
distinct_pincodes = df['Pincode'].nunique()
distinct_locations = df['Location'].nunique()

print(f"Number of distinct Pincodes: {distinct_pincodes}")
print(f"Number of distinct Locations: {distinct_locations}")

# Also showing general info
df.info()

Number of distinct Pincodes: 67
Number of distinct Locations: 228
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 229 entries, 0 to 228
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Location          228 non-null    object 
 1   Pincode           228 non-null    float64
 2   State             228 non-null    object 
 3   District          228 non-null    object 
 4   Category          229 non-null    object 
 5   Normalized Score  229 non-null    float64
dtypes: float64(2), object(4)
memory usage: 10.9+ KB


In [ ]:
# Convert the DataFrame to a CSV file
df.to_csv('data/raw/hyderabad_pincodes.csv', index=False)
print("DataFrame successfully converted to 'data/raw/hyderabad_pincodes.csv'")

DataFrame successfully converted to 'data/raw/hyderabad_pincodes.csv'


In [ ]:
# Grouping localities by pincode for easier categorization
pincode_groups = df.groupby('Pincode')['Location'].apply(lambda x: ', '.join(x.dropna().astype(str).unique())).reset_index()

# Updated mapping function with 'Low Density/Outskirts'
def categorize_pincode(row):
    p = int(row['Pincode'])
    locs = row['Location'].lower()

    # IT Hubs
    if p in [500081, 500032, 500019, 500084, 500089]:
        return 'IT'
    # Old City
    elif p in [500002, 500065, 500023, 500053, 500052, 500064, 500002]:
        return 'Old City'
    # Dense / Commercial / Central
    elif p in [500001, 500095, 500012, 500003, 500027, 500004]:
        return 'Dense'
    # Low Density / Outskirts (Commonly higher pincode ranges or specific peripheral areas)
    elif p >= 501000 or any(word in locs for word in ['village', 'shamshabad', 'medchal', 'ghatkesar', 'moula ali']):
        return 'Low Density/Outskirts'
    # General Residential
    elif any(word in locs for word in ['colony', 'hills', 'nagar', 'vihar', 'enclave']):
        return 'Residential'
    else:
        return 'Residential'

pincode_groups['Category'] = pincode_groups.apply(categorize_pincode, axis=1)

# Re-merge to update the main dataframe
if 'Category' in df.columns:
    df = df.drop(columns=['Category'])
df = df.merge(pincode_groups[['Pincode', 'Category']], on='Pincode', how='left')

print("Updated categorization summary:")
print(pincode_groups['Category'].value_counts())
display(pincode_groups.head(20))

Updated categorization summary:
Category
Residential              51
Dense                     6
Old City                  6
Low Density/Outskirts     3
IT                        1
Name: count, dtype: int64


,Pincode,Location,Category
0,500001.0,"Gandhi Bhawan, Hindi Bhawan, Hyderabad., Moazz...",Dense
1,500002.0,"Hyderabad Jubilee, Moghalpura, Tagarikanaka",Old City
2,500003.0,"Begumpet Policelines, Kingsway, Mg Road, Picke...",Dense
3,500004.0,"A.Gs office, Anandnagar, Bazarghat, Khairataba...",Dense
4,500005.0,"Balapur, Crp Camp, Keshogiri, Mamidipalli, Pah...",Residential
5,500006.0,"Dhoolpet, Karwan Sahu, Mangalhat",Residential
6,500007.0,"Administrative Buildings, Iict, Jama I osmania...",Residential
7,500008.0,"Dargah Hussain shahwali, Golconda, Hyder Shah ...",Residential
8,500012.0,"Afzalgunj, Begumbazar, Osmania General hospital",Dense
9,500013.0,"Amberpet, Central Police lines, Hyderabad Publ...",Residential


In [ ]:
import numpy as np

def assign_weight(area_type):
    if area_type == "IT":
        return np.random.uniform(0.85, 1.0)
    elif area_type == "Dense" or area_type == "Old City":
        return np.random.uniform(0.75, 0.95)
    elif area_type == "Residential":
        return np.random.uniform(0.5, 0.75)
    else: # Low Density / Outskirts
        return np.random.uniform(0.3, 0.5)

# Applying the weight assignment to the main DataFrame
df['Population Density'] = df['Category'].apply(assign_weight)

print("Population Density weights assigned successfully!")
display(df.head(10))

Population Density weights assigned successfully!


,Location,Pincode,State,District,Normalized Score,Category,Population Density
0,A.G.college,500030.0,Telangana,Hyderabad,0.812458,Residential,0.585236
1,A.Gs office,500004.0,Telangana,Hyderabad,0.758359,Dense,0.776240
2,A.Gs. staff quarters,500045.0,Telangana,Hyderabad,0.922575,Residential,0.612885
3,Administrative Buildings,500007.0,Telangana,Hyderabad,0.451864,Residential,0.575272
4,Afzalgunj,500012.0,Telangana,Hyderabad,0.535010,Dense,0.782825
5,Aliabad,500015.0,Telangana,Hyderabad,0.785780,Residential,0.594536
6,Ambernagar,500044.0,Telangana,Hyderabad,0.872626,Residential,0.520280
7,Amberpet,500013.0,Telangana,Hyderabad,0.842977,Residential,0.621771
8,Anandnagar,500004.0,Telangana,Hyderabad,0.777342,Dense,0.893914
9,Anantagiri,501201.0,Telangana,Hyderabad,0.150276,Low Density/Outskirts,0.355741


In [ ]:
import numpy as np

# Ensure Location column is string to avoid AttributeError
df['Location'] = df['Location'].astype(str)

# Define infrastructure scores based on user input
infra_mapping = {
    'Old City': 0.6,
    'IT': 0.85,
    'Dense': 0.7,
    'Residential': 0.75,
    'Low Density/Outskirts': 0.65
}

# Specific Boost for high-profile IT areas
special_boost_locations = ['gachibowli', 'madhapur', 'hitec city', 'kondapur']

def calculate_instability(row):
    # Base infra score from category
    base_score = infra_mapping.get(row['Category'], 0.7)

    # Check for Special Boost areas in the location string (now safe as string)
    loc_lower = row['Location'].lower()
    is_boosted = any(area in loc_lower for area in special_boost_locations)

    # Instability = (1 - infra_score) + random noise
    instability = (1 - base_score) + np.random.uniform(0, 0.15)

    if is_boosted:
        instability += 0.2

    return round(instability, 3)

# Map infra score
df['Infra Score'] = df['Category'].map(infra_mapping)

# Calculate Instability Factor
df['Instability Factor'] = df.apply(calculate_instability, axis=1)

print("Advanced Realism factors added successfully!")
display(df.sort_values(by='Instability Factor', ascending=False).head(10))

Advanced Realism factors added successfully!


,Location,Pincode,State,District,Category,Population Density,Infra Score,Instability Factor
187,Shahalibanda,500065.0,Telangana,Hyderabad,Old City,0.861676,0.60,0.548
103,Kishanbagh,500064.0,Telangana,Hyderabad,Old City,0.755150,0.60,0.542
189,Shivarampalli,500052.0,Telangana,Hyderabad,Old City,0.889487,0.60,0.533
206,Tadbun,500064.0,Telangana,Hyderabad,Old City,0.875882,0.60,0.524
21,Bahadurpura,500064.0,Telangana,Hyderabad,Old City,0.794712,0.60,0.522
36,Chandulalbaradari,500064.0,Telangana,Hyderabad,Old City,0.930591,0.60,0.518
207,Tagarikanaka,500002.0,Telangana,Hyderabad,Old City,0.805799,0.60,0.512
172,Reinbazar,500023.0,Telangana,Hyderabad,Old City,0.757715,0.60,0.508
204,Svpnpa,500052.0,Telangana,Hyderabad,Old City,0.754762,0.60,0.505
12,Ankireddipalli,501301.0,Telangana,Hyderabad,Low Density/Outskirts,0.428731,0.65,0.498


In [ ]:
# Calculate Infrastructure Adjustment
df['Infra Adjustment'] = (1 + (1 - df['Infra Score']))

# Calculate Final Weight using the recommended formula
df['Final Weight'] = df['Population Density'] * (1 + df['Instability Factor']) * df['Infra Adjustment']

# Normalize the score between 0 and 1
min_w = df['Final Weight'].min()
max_w = df['Final Weight'].max()

df['Normalized Score'] = (df['Final Weight'] - min_w) / (max_w - min_w)

print("Updated Final Weights using recommended formulas and Normalized Scores!")
display(df.sort_values(by='Normalized Score', ascending=False).head(10))

Updated Final Weights using recommended formulas and Normalized Scores!


,Location,Pincode,State,District,Category,Population Density,Infra Score,Instability Factor,Final Weight,Normalized Score,Infra Adjustment
36,Chandulalbaradari,500064.0,Telangana,Hyderabad,Old City,0.930591,0.6,0.518,1.977693,1.000000,1.4
214,Uppuguda,500053.0,Telangana,Hyderabad,Old City,0.942630,0.6,0.450,1.913540,0.952501,1.4
189,Shivarampalli,500052.0,Telangana,Hyderabad,Old City,0.889487,0.6,0.533,1.909018,0.949153,1.4
206,Tadbun,500064.0,Telangana,Hyderabad,Old City,0.875882,0.6,0.524,1.868781,0.919363,1.4
187,Shahalibanda,500065.0,Telangana,Hyderabad,Old City,0.861676,0.6,0.548,1.867423,0.918357,1.4
76,Hyderabad Jubilee,500002.0,Telangana,Hyderabad,Old City,0.894030,0.6,0.474,1.844920,0.901695,1.4
159,Quazipura,500065.0,Telangana,Hyderabad,Old City,0.899870,0.6,0.426,1.796500,0.865846,1.4
50,Falaknuma,500053.0,Telangana,Hyderabad,Old City,0.861419,0.6,0.466,1.767976,0.844727,1.4
64,Hasannagar,500052.0,Telangana,Hyderabad,Old City,0.869970,0.6,0.426,1.736807,0.821650,1.4
207,Tagarikanaka,500002.0,Telangana,Hyderabad,Old City,0.805799,0.6,0.512,1.705715,0.798629,1.4


In [ ]:
import pandas as pd
import numpy as np
import random

# 1. Create hourly date range for 2 years
dates = pd.date_range(start="2023-01-01", end="2024-12-31", freq="h")
time_df = pd.DataFrame({"timestamp": dates})

# 2. Add time-based features
time_df['hour'] = time_df['timestamp'].dt.hour
time_df['dayofweek'] = time_df['timestamp'].dt.dayofweek
time_df['month'] = time_df['timestamp'].dt.month
time_df['date_str'] = time_df['timestamp'].dt.strftime('%Y-%m-%d')

# 3. Define time weights function
def calculate_time_weight(row):
    weight = 1.0
    if 18 <= row['hour'] <= 23: weight *= 1.5
    if row['dayofweek'] >= 5: weight *= 1.3
    if row['month'] in [7, 8, 9]: weight *= 1.7
    return weight

time_df['Time Weight'] = time_df.apply(calculate_time_weight, axis=1)

# 4. Define Outage Events (Manual + Randomly Generated)
outages = [
    {"area": "Gachibowli", "date": "2024-07-15"},
    {"area": "Madhapur", "date": "2024-08-02"},
    {"area": "Kukatpally", "date": "2023-05-10"}
]

# Generate 10 additional random outages
all_locations = df['Location'].unique().tolist()
all_dates = time_df['date_str'].unique().tolist()

for _ in range(10):
    random_loc = random.choice(all_locations)
    random_date = random.choice(all_dates)
    outages.append({"area": str(random_loc), "date": random_date})

print("Time Engine updated with 10 additional random outage events!")
print(f"Total time steps: {len(time_df)}")
print(f"Total Outage Events: {len(outages)}")
display(time_df.head())
print("\nSample of Outage Events:")
for o in outages[:8]:
    print(f"- {o['area']} on {o['date']}")

Time Engine updated with 10 additional random outage events!
Total time steps: 17521
Total Outage Events: 13


,timestamp,hour,dayofweek,month,date_str,Time Weight
0,2023-01-01 00:00:00,0,6,1,2023-01-01,1.3
1,2023-01-01 01:00:00,1,6,1,2023-01-01,1.3
2,2023-01-01 02:00:00,2,6,1,2023-01-01,1.3
3,2023-01-01 03:00:00,3,6,1,2023-01-01,1.3
4,2023-01-01 04:00:00,4,6,1,2023-01-01,1.3



Sample of Outage Events:
- Gachibowli on 2024-07-15
- Madhapur on 2024-08-02
- Kukatpally on 2023-05-10
- Hyd Airport ii on 2023-02-17
- Ankushapur on 2023-07-23
- Aperl on 2023-06-06
- Kolthur on 2023-06-13
- Mamidipalli on 2024-03-17


In [ ]:
import numpy as np
import random

# 1. Expanded Complaint Types
complaint_types = {
    "No Internet": 0.35,
    "Slow Speed": 0.20,
    "Intermittent": 0.15,
    "High Latency": 0.10,
    "Packet Loss": 0.08,
    "Router Issue": 0.07,
    "DNS Issue": 0.05
}

# 2. Enhanced Grievance Engine (Templates for 500+ variations)
grievance_templates = {
    "No Internet": [
        "Net down since morning.", "intrnet not wrkng.", "Red light on router.",
        "Internet not working in my area.", "Facing total blackout of internet.",
        "Bhai net nahi chal raha.", "Internet cut ho gaya.", "Total signal loss.",
        "Urgent: No internet connection.", "Work from home affected, fix net."
    ],
    "Slow Speed": [
        "Speed is very slw today.", "Streaming buffering constant.", "Not getting promised speed.",
        "Very slow internet, taking hours to load.", "Speeds have dropped drastically.",
        "Net bohot slow hai.", "Downloading is impossible.", "Speed test showing 1mbps only."
    ],
    "Intermittent": [
        "Connection drops every 10 mins.", "Unstable wifi connection.", "Net coming and going.",
        "Disconnecting frequently during calls.", "Wifi signal unstable.", "Bar bar cut raha hai."
    ],
    "High Latency": [
        "Ping is too high for gaming.", "Lagging during video calls.", "Response time is very poor.",
        "High latency issues since evening.", "Games are unplayable due to lag."
    ],
    "Packet Loss": [
        "Facing heavy packet loss.", "Data packets dropping constantly.", "Voice breaking on calls.",
        "Heavy loss detected in network path.", "Zoom calls are stuttering."
    ],
    "Router Issue": [
        "Router getting overheated.", "Power light not blinking.", "Router needs replacement.",
        "Device not turning on.", "Adapter issue with the router."
    ],
    "DNS Issue": [
        "DNS probe finished nxdomain error.", "Cannot resolve site addresses.", "DNS server not responding.",
        "Site not loading but ping works.", "Changing DNS didn't help."
    ]
}

def get_complaint_type(location, date_str):
    is_outage = False
    for event in outages:
        if event['date'] == date_str and event['area'].lower() in location.lower():
            is_outage = True
            break

    if is_outage:
        types = ["No Internet", "Slow Speed", "Intermittent", "Router Issue"]
        probs = [0.8, 0.1, 0.05, 0.05]
        return np.random.choice(types, p=probs)
    else:
        types = list(complaint_types.keys())
        probs = list(complaint_types.values())
        return np.random.choice(types, p=probs)

def generate_enhanced_grievance(comp_type):
    # Pick a base template
    base = random.choice(grievance_templates.get(comp_type, ["Internet issue."]))
    # Randomly capitalize for urgency
    if random.random() < 0.1: base = base.upper()
    return base

print("Upgraded Complaint & Grievance Engines Initialized!")

Upgraded Complaint & Grievance Engines Initialized!


In [ ]:
!pip install faker -q
from faker import Faker
import random

# Initialize Faker for India
fake = Faker('en_IN')

# Step 5: Address Generator
def generate_full_address(location, pincode):
    # Generate a fake base address (House No, Street)
    base_address = fake.address().split('\n')[0]
    # Append specific Area and Pincode from our dataset
    return f"{base_address}, {location}, Hyderabad, {int(pincode)}"

# Step 4: Grievance Text Engine (Base Logic)
def generate_grievance_text(complaint_type):
    # We will expand this with 200-500 variations/LLM later
    # For now, providing a simple mapping as a foundation
    grievance_map = {
        "No Internet": ["My internet is down since morning.", "Internet not working at home.", "No signal on router."],
        "Slow Speed": ["Speed is very slow.", "Streaming is buffering constantly.", "Not getting the promised speed."],
        "Intermittent": ["Connection keeps dropping.", "Wifi connects and disconnects every 5 mins."],
        "Router Issue": ["Router lights are red.", "Device not turning on."]
    }
    return random.choice(grievance_map.get(complaint_type, ["Internet issue."]))

print("Address Generator and Grievance Engine Base Initialized!")

# Test the generators
sample_loc = "Gachibowli"
sample_pin = 500032
print(f"Sample Address: {generate_full_address(sample_loc, sample_pin)}")
print(f"Sample Grievance: {generate_grievance_text('No Internet')}")

   â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 2.0/2.0 MB 38.5 MB/s eta 0:00:00
Address Generator and Grievance Engine Base Initialized!
Sample Address: 10, Thakkar Road, Ambattur 598840, Gachibowli, Hyderabad, 500032
Sample Grievance: My internet is down since morning.


In [ ]:
import uuid
import re
import pandas as pd
import numpy as np
from datetime import timedelta

In [ ]:
TOTAL_COMPLAINTS = 50000
RECURRING_CUST_PROB = 0.15 # 15% chance complaint comes from an existing customer

complaints_data = []
customer_base = [] # List of (customer_id, address, pincode, location, category)

# Probability Distributions
area_probs = df['Normalized Score'] / df['Normalized Score'].sum()

print(f"Starting Enhanced Generation of {TOTAL_COMPLAINTS} records...")

for i in range(TOTAL_COMPLAINTS):
    # 1. Determine if this is a recurring or new customer
    if customer_base and random.random() < RECURRING_CUST_PROB:
        cust = random.choice(customer_base)
        cust_id, full_addr, pin, loc, cat = cust
    else:
        # Create new customer profile
        selected_row = df.sample(n=1, weights=area_probs).iloc[0]
        loc = selected_row['Location']
        pin = selected_row['Pincode']
        cat = selected_row['Category']
        cust_id = f"CUST-{random.randint(100000, 999999)}"
        full_addr = generate_full_address(loc, pin)
        customer_base.append((cust_id, full_addr, pin, loc, cat))

    # 2. Sample Timestamp with jitter for realism (Adding random minutes/seconds)
    timestamp_row = time_df.sample(n=1, weights='Time Weight').iloc[0]
    base_ts = timestamp_row['timestamp']
    jitter = timedelta(minutes=random.randint(0, 59), seconds=random.randint(0, 59))
    ts = base_ts + jitter

    # 3. Generate content
    comp_type = get_complaint_type(loc, timestamp_row['date_str'])
    grievance = generate_enhanced_grievance(comp_type)
    grievance = inject_typos(grievance)

    complaints_data.append({
        "ticket_id": str(uuid.uuid4())[:8].upper(),
        "customer_id": cust_id,
        "timestamp": ts,
        "location": loc,
        "pincode": int(pin),
        "category": cat,
        "complaint_type": comp_type,
        "grievance_text": grievance,
        "customer_address": full_addr
    })

final_df = pd.DataFrame(complaints_data).sort_values(by='timestamp')
display(final_df.head(10))

print(f"\nTotal Records: {len(final_df)}")
print(f"Unique Customers: {final_df['customer_id'].nunique()}")
final_df.to_csv('hyderabad_isp_complaints_v2_pro.csv', index=False)
print("Saved as hyderabad_isp_complaints_v2_pro.csv")

Starting Enhanced Generation of 50000 records...


,ticket_id,customer_id,timestamp,location,pincode,category,complaint_type,grievance_text,customer_address
44947,5626175C,CUST-265888,2023-01-01 01:57:22,Padmavathi Nagar,500004,Dense,No Internet,Internet not wrkng in my area.,"920, Padmavathi Nagar, Hyderabad, 500004"
22044,4CA0B1AA,CUST-708992,2023-01-01 02:32:55,Ibrahim Bagh lines,500031,Residential,Slow Speed,Net bohot slow hai.,"H.No. 16, Rastogi Zila, Ibrahim Bagh lines, Hy..."
15273,17B292B2,CUST-185245,2023-01-01 02:58:04,Pahadishareef,500005,Residential,Slow Speed,STREAMING BUFFERING CONSTANT.,"114, Sura, Silchar-383938, Pahadishareef, Hyde..."
27731,3C071CAC,CUST-263061,2023-01-01 03:50:29,Saidabad,500059,Residential,Packet Loss,Zoom calls are stuttering.,"H.No. 85, Bhakta Circle, Saidabad, Hyderabad, ..."
14526,3E97A043,CUST-821971,2023-01-01 04:15:28,State Bank of india,500095,Dense,Intermittent,Net coming and going.,"H.No. 04, Patla Chowk, State Bank of india, Hy..."
21337,C3CC3821,CUST-284322,2023-01-01 04:21:12,New Mla quarters,500063,Residential,No Internet,Facing total blackout of internet.,"H.No. 96, Palla Street, Thrissur 138025, New M..."
41970,3D2D0E88,CUST-946282,2023-01-01 04:28:37,Yousufguda,500045,Residential,No Internet,Red light on router.,"907, Yousufguda, Hyderabad, 500045"
44052,FEF43C21,CUST-644738,2023-01-01 04:30:47,Lallaguda,500017,Residential,Intermittent,Connection drops every 10 mins.,"64/12, Agarwal Street, Lallaguda, Hyderabad, 5..."
8641,465367D8,CUST-197961,2023-01-01 04:37:33,Central Secretariat,500022,Residential,No Internet,INTRNET NOT WRKNG.,"12/51, Dhar Chowk, Central Secretariat, Hydera..."
10546,41563F6C,CUST-883300,2023-01-01 05:10:25,Begumpet Policelines,500003,Dense,Intermittent,Disconnecting frequently during calls.,"H.No. 42, Sibal Circle, Begumpet Policelines, ..."



Total Records: 50000
Unique Customers: 41463
Saved as hyderabad_isp_complaints_v2_pro.csv


In [ ]:
# Export the massive dataset to CSV
final_df.to_csv('data/raw/hyderabad_isp_complaints_v1.csv', index=False)
print("Master dataset exported to 'data/raw/hyderabad_isp_complaints_v1.csv'")

Master dataset exported to 'data/raw/hyderabad_isp_complaints_v1.csv'


In [ ]:
# Calculating unique values for each column in the final dataset
unique_counts = final_df.nunique()

print("Unique values in each column (Total Records: 50,000):")
print(unique_counts)

Unique values in each column (Total Records: 50,000):
ticket_id           50000
timestamp           16143
location              227
pincode                67
category                5
complaint_type          4
grievance_text         20
customer_address    39334
dtype: int64


In [ ]:
# Calculating unique values for each column in the V2 Pro dataset
unique_counts_v2 = final_df.nunique()

print("Unique values in each column (V2 Pro - Total Records: 50,000):")
print(unique_counts_v2)

Unique values in each column (V2 Pro - Total Records: 50,000):
ticket_id           50000
customer_id         41463
timestamp           49967
location              227
pincode                67
category                5
complaint_type          7
grievance_text        124
customer_address    41679
dtype: int64


In [ ]:
!pip install pandas numpy faker openai tqdm -q
import pandas as pd
import numpy as np
import random
from faker import Faker
from tqdm import tqdm
import uuid
import datetime
from datetime import timedelta

# Configuration
N = 50000
USE_LLM = False
fake = Faker('en_IN')

# Fix global variable to match function expectation
complaint_types = {
    "No Internet": 0.35,
    "Slow Speed": 0.20,
    "Intermittent": 0.15,
    "High Latency": 0.10,
    "Packet Loss": 0.08,
    "Router Issue": 0.07,
    "DNS Issue": 0.05
}

# --- Engines ---
def get_severity(complaint_type):
    if complaint_type == "No Internet": return random.choices(["High", "Medium"], [0.7, 0.3])[0]
    elif complaint_type in ["Slow Speed", "Intermittent"]: return random.choices(["Medium", "Low"], [0.6, 0.4])[0]
    else: return random.choice(["Low", "Medium"])

def get_language_style(area_category):
    cat_str = str(area_category)
    if "IT" in cat_str: return random.choices(["english", "mixed"], [0.8, 0.2])[0]
    elif "Dense" in cat_str or "Old City" in cat_str: return random.choices(["hinglish", "urdu_mix"], [0.7, 0.3])[0]
    else: return random.choice(["english", "hinglish"])

def fallback_text(ctype, tone, lang):
    base = {
        "No Internet": ["internet not working", "no connection", "wifi down"],
        "Slow Speed": ["internet very slow", "speed is low", "buffering issue"],
        "Intermittent": ["disconnecting again and again", "unstable connection"],
        "High Latency": ["ping too high", "lag issue"],
        "Packet Loss": ["packet loss happening", "data loss issue"],
        "Router Issue": ["router not working", "router issue"],
        "DNS Issue": ["dns not resolving", "cannot access sites"]
    }
    text = random.choice(base[ctype])
    if tone == "angry": text = text.upper() + "!!!"
    elif tone == "polite": text = "please check, " + text
    elif tone == "frustrated": text += " since long time"

    flavor = {"hinglish": " yaar", "urdu_mix": " bhai", "mixed": " pls fix"}
    return text + flavor.get(lang, "")

def inject_noise(text):
    if random.random() < 0.15: text = text.replace("internet", "intrnet")
    if random.random() < 0.1: text = text.replace("working", "wrkng")
    return text

customer_pool = []
def get_customer(loc, pin):
    if random.random() < 0.15 and customer_pool:
        return random.choice(customer_pool)
    else:
        cust_id = f"CUST-{random.randint(100000, 999999)}"
        address = generate_full_address(loc, pin)
        customer_pool.append((cust_id, address))
        return cust_id, address

# --- Main Loop ---
data = []
area_weights = df['Normalized Score'].fillna(0.0001)

for i in tqdm(range(N), desc="Generating Complaints", unit="comp"):
    loc_row = df.sample(n=1, weights=area_weights).iloc[0]

    # V2 Technique: Sample from time_df and add jitter
    time_row = time_df.sample(n=1, weights='Time Weight').iloc[0]
    base_ts = time_row['timestamp']
    jitter = timedelta(minutes=random.randint(0, 59), seconds=random.randint(0, 59))
    ts = base_ts + jitter
    date_str = time_row['date_str']

    # Generate complaint type
    ctype = get_complaint_type(location, date_str)

    severity = get_severity(ctype)
    lang = get_language_style(loc_row['Category'])
    tone = random.choice(["normal", "frustrated", "angry", "polite"])
    text = inject_noise(fallback_text(ctype, tone, lang))
    cust_id, address = get_customer(loc_row['Location'], loc_row['Pincode'])

    data.append({
        "ticket_id": f"TKT{i+1:06d}",
        "customer_id": cust_id,
        "timestamp": ts,
        "location": loc_row['Location'],
        "pincode": int(loc_row['Pincode']),
        "category": loc_row['Category'],
        "complaint_type": ctype,
        "severity": severity,
        "grievance_text": text,
        "customer_address": address
    })

final_v3_df = pd.DataFrame(data).sort_values("timestamp")
final_v3_df.to_csv("synthetic_complaints_v3.csv", index=False)
print("\nDataset V3 with High-Resolution Timestamps generated successfully!")
display(final_v3_df.head())

Generating Complaints: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 50000/50000 [01:48<00:00, 458.91comp/s]



Dataset V3 with High-Resolution Timestamps generated successfully!


,ticket_id,customer_id,timestamp,location,pincode,category,complaint_type,severity,grievance_text,customer_address
8555,TKT008556,CUST-374408,2023-01-01 00:16:23,Hyderabad Public school,500013,Residential,High Latency,Low,LAG ISSUE!!!,"H.No. 89, Mistry Zila, Berhampur 397949, Vidha..."
34885,TKT034886,CUST-505870,2023-01-01 00:33:56,A.G.college,500030,Residential,No Internet,High,INTERNET NOT WORKING!!! yaar,"H.No. 75, Warrior Zila, New Delhi-777594, A.G...."
6951,TKT006952,CUST-471958,2023-01-01 00:42:02,Musheerabad Dso,500020,Residential,Slow Speed,Medium,"please check, buffering issue","93, Buch Street, Musheerabad Dso, Hyderabad, 5..."
26631,TKT026632,CUST-695627,2023-01-01 00:54:57,Pragatinagar,500015,Residential,High Latency,Medium,"please check, ping too high","H.No. 078, Pragatinagar, Hyderabad, 500015"
28676,TKT028677,CUST-846737,2023-01-01 02:08:02,Hindi Bhawan,500001,Dense,High Latency,Low,ping too high bhai,"H.No. 212, Bazarghat, Hyderabad, 500004"


In [ ]:
# Calculating unique values for each column in the V3 dataset
unique_counts_v3 = final_v3_df.nunique()

print("Unique values in each column (V3 - Total Records: 50,000):")
print(unique_counts_v3)

Unique values in each column (V3 - Total Records: 50,000):
ticket_id           50000
customer_id         41511
timestamp           49980
location              227
pincode                67
category                5
complaint_type          7
severity                3
grievance_text        286
customer_address    41753
dtype: int64


In [ ]:
!pip install faker tqdm -q
import pandas as pd
import numpy as np
import random
from faker import Faker
from tqdm import tqdm
import uuid
import datetime
from datetime import timedelta

# Configuration
N = 50000
fake = Faker('en_IN')

# Using the dataframe 'df' that was scraped in the first cell
# If it's missing the expected 'Normalized Score', we'll generate it here
if 'df' not in globals():
    print("Warning: Base dataframe 'df' not found. Please run the first cell first.")
else:
    if 'Normalized Score' not in df.columns:
        df['Normalized Score'] = np.random.uniform(0.1, 1.0, len(df))

    complaint_types = {
        "No Internet": 0.35,
        "Slow Speed": 0.20,
        "Intermittent": 0.15,
        "High Latency": 0.10,
        "Packet Loss": 0.08,
        "Router Issue": 0.07,
        "DNS Issue": 0.05
    }

    def generate_full_address(location, pincode):
        base_address = fake.address().split('\n')[0]
        return f"{base_address}, {location}, Hyderabad, {int(pincode)}"

    def fallback_text(ctype, tone, lang):
        base_phrases = {
            "No Internet": ["internet not working", "no connection", "wifi down", "internet gone"],
            "Slow Speed": ["internet very slow", "speed is low", "buffering issue", "net slow"],
            "Intermittent": ["disconnecting again and again", "unstable connection", "keeps dropping"],
            "High Latency": ["ping too high", "lag issue", "delay in response"],
            "Packet Loss": ["packet loss happening", "data loss issue"],
            "Router Issue": ["router not working", "router problem"],
            "DNS Issue": ["dns not resolving", "cannot access sites"]
        }
        patterns = ["{issue}", "{issue} since morning", "{issue} pls fix", "facing {issue}", "still {issue}"]
        issue = random.choice(base_phrases[ctype])
        sentence = random.choice(patterns).format(issue=issue)
        if tone == "angry": sentence = sentence.upper() + "!!!"
        if lang == "hinglish": sentence += " yaar"
        return sentence

    def get_language_style(area_category):
        cat_str = str(area_category)
        if "IT" in cat_str: return "english"
        return "hinglish"

    customer_pool = []
    def get_customer(loc, pin):
        if random.random() < 0.15 and customer_pool:
            return random.choice(customer_pool)
        else:
            cust_id = f"CUST-{random.randint(100000, 999999)}"
            address = generate_full_address(loc, pin)
            customer_pool.append((cust_id, address))
            return cust_id, address

    data = []
    area_weights = df['Normalized Score'].fillna(0.0001)

    # Re-creating a basic time engine if not found
    dates = pd.date_range(start="2023-01-01", end="2024-12-31", freq="h")
    time_engine = pd.DataFrame({"timestamp": dates})

    for i in tqdm(range(N), desc="Generating Complaints V4"):
        loc_row = df.sample(n=1, weights=area_weights).iloc[0]
        time_row = time_engine.sample(n=1).iloc[0]
        ts = time_row['timestamp'] + timedelta(minutes=random.randint(0, 59))

        ctype = np.random.choice(list(complaint_types.keys()), p=list(complaint_types.values()))
        lang = get_language_style(loc_row.get('Category', 'Residential'))
        tone = random.choice(["normal", "angry", "polite"])
        text = fallback_text(ctype, tone, lang)

        cust_id, address = get_customer(loc_row['Location'], loc_row['Pincode'])

        data.append({
            "ticket_id": f"TKT{i+1:06d}",
            "customer_id": cust_id,
            "timestamp": ts,
            "location": loc_row['Location'],
            "pincode": int(loc_row['Pincode']),
            "complaint_type": ctype,
            "grievance_text": text,
            "customer_address": address
        })

    final_v4_df = pd.DataFrame(data).sort_values("timestamp")
    final_v4_df.to_csv("data/raw/synthetic_complaints_v4.csv", index=False)
    display(final_v4_df.head())

In [ ]:
# Calculating unique values for each column in the V4 dataset
unique_counts_v4 = final_v4_df.nunique()

print("Unique values in each column (V4 - Total Records: 50,000):")
print(unique_counts_v4)

Unique values in each column (V4 - Total Records: 50,000):
ticket_id           50000
customer_id         41538
timestamp           49976
location              227
pincode                67
category                5
complaint_type          7
severity                3
grievance_text       8248
customer_address    41769
dtype: int64


In [ ]:
import pandas as pd

# Load the data/raw/train_data.csv file
train_df = pd.read_csv('data/raw/train_data.csv')

# Calculate unique values for each column
unique_counts_train = train_df.nunique()

print("Unique values in each column of data/raw/train_data.csv:")
print(unique_counts_train)

Unique values in each column of data/raw/train_data.csv:
grievance_text    7020
severity             2
dtype: int64


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

def assign_severity_level(text):
    text_str = str(text)
    text_lower = text_str.lower()

    # Severity Markers
    urgency = any(w in text_lower for w in ["urgent", "asap", "immediately"])
    long_duration = any(w in text_lower for w in ["since morning", "since night", "long time", "2 days"])
    failure = any(w in text_lower for w in ["not working", "down", "no connection", "wifi down", "internet gone"])
    frustration = "!!!" in text_str or text_str.isupper()
    repetition = any(w in text_lower for w in ["again", "continuously", "repeatedly"])

    # Mapping based on user request:
    # High1=1, High2=2, High3=3, Low1=4, Low2=5
    if failure:
        if long_duration and (urgency or frustration):
            return 3 # High3
        elif long_duration or urgency or frustration:
            return 2 # High2
        return 1     # High1

    is_slow = "slow" in text_lower or "lag" in text_lower or "buffering" in text_lower

    if is_slow:
        # Mapping other Lows to 5 (Low2) for simplicity or as highest Low severity
        return 5

    return 4 # Low1

# Prepare data
df_processed = train_df.copy()
df_processed['severity_rank'] = df_processed['grievance_text'].apply(assign_severity_level)

# Split
train_set, test_set = train_test_split(df_processed, test_size=0.2, random_state=42, stratify=df_processed['severity_rank'])

# Vectorize
vectorizer = TfidfVectorizer(max_features=8000, ngram_range=(1,2), stop_words='english')
X_train = vectorizer.fit_transform(train_set['grievance_text'])
X_test = vectorizer.transform(test_set['grievance_text'])

# Train (using numerical ranks directly)
y_train = train_set['severity_rank']
y_test = test_set['severity_rank']

model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

# Results
y_pred = model.predict(X_test)
print("Numerical Severity Classification Report (1-5):\n")
print(classification_report(y_test, y_pred))

# Export updated processed df for the download cell
df = df_processed

Numerical Severity Classification Report (1-5):

              precision    recall  f1-score   support

           1       0.78      0.72      0.75       895
           2       0.75      0.83      0.79      1058
           3       0.75      0.11      0.19        54
           4       1.00      1.00      1.00      2597
           5       1.00      1.00      1.00       783

    accuracy                           0.91      5387
   macro avg       0.86      0.73      0.75      5387
weighted avg       0.91      0.91      0.91      5387



In [ ]:
from google.colab import files

# 'df' contains the grievance_text and the new granular 'severity_rank' column
df.to_csv('data/raw/categorized_complaints_v6.csv', index=False)

print("File 'data/raw/categorized_complaints_v6.csv' has been created.")
files.download('data/raw/categorized_complaints_v6.csv')

File 'data/raw/categorized_complaints_v6.csv' has been created.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
pip install faker

   â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 2.0/2.0 MB 32.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import random
import requests
import io
from faker import Faker
from tqdm import tqdm
from datetime import timedelta

N = 50000
fake = Faker('en_IN')

# --- Step 1: Re-initialize Base Data (df) ---
print("Initializing base location data...")
url = "https://www.bankbazaar.com/pin-code/telangana/hyderabad.html"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)
tables = pd.read_html(io.StringIO(response.text))
df = tables[0]

# --- FIX: Clean NaN values from Pincode/Location before processing ---
df = df.dropna(subset=['Pincode', 'Location'])
df['Pincode'] = df['Pincode'].astype(int)
df['Normalized Score'] = np.random.uniform(0.1, 1.0, len(df))

# --- Step 2: Re-initialize Time Engine (time_df) ---
print("Initializing time engine...")
dates = pd.date_range(start="2023-01-01", end="2024-12-31", freq="h")
time_df = pd.DataFrame({"timestamp": dates})

# --- Step 3: Severity Engine Configuration ---
severity_templates = {
    1: ["TOTAL BLACKOUT. {net} NOT WORKING. FIX ASAP!!", "URGENT: No signal. Work affected. Fix immediately.", "{net} down since 2 days. This is unacceptable!!"],
    2: ["{net} is not working again.", "Facing total connection failure since morning.", "Why is my {net} down? Please resolve."],
    3: ["{net} keeps dropping every 10 minutes.", "Very slow speeds for the last 5 hours.", "Internet is intermittent and unstable."],
    4: ["Speed is a bit slow today.", "Watching videos is buffering sometimes.", "Ping is high in gaming, check please."],
    5: ["I want to check my data balance.", "Please tell me about new plans.", "Is there any maintenance today? Thank you."]
}

def generate_v5_text(rank):
    base = random.choice(severity_templates[rank])
    net_val = random.choice(["internet", "wifi", "connection", "net"])
    text = base.format(net=net_val)
    if random.random() < 0.1: text = text.replace("internet", "intrnet").replace("WORKING", "WRKNG")
    return text

# --- Step 4: Generation Loop ---
data = []
area_weights = df['Normalized Score'].fillna(0.0001)

for i in tqdm(range(N), desc="Generating RNN-Ready Dataset"):
    loc_row = df.sample(n=1, weights=area_weights).iloc[0]
    time_row = time_df.sample(n=1).iloc[0]
    ts = time_row['timestamp'] + timedelta(minutes=random.randint(0, 59))

    rank = np.random.choice([1, 2, 3, 4, 5], p=[0.15, 0.25, 0.30, 0.20, 0.10])
    text = generate_v5_text(rank)

    data.append({
        "ticket_id": f"TKT{i+1:06d}",
        "timestamp": ts,
        "location": loc_row['Location'],
        "pincode": int(loc_row['Pincode']),
        "severity_rank": rank,
        "grievance_text": text
    })

rnn_ready_df = pd.DataFrame(data).sort_values("timestamp")
rnn_ready_df.to_csv("rnn_ready_complaints_v6.csv", index=False)
print("\nDataset generated successfully: rnn_ready_complaints_v6.csv")
display(rnn_ready_df.head())


/tmp/ipykernel_6474/954433688.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Pincode'] = df['Pincode'].astype(int)
/tmp/ipykernel_6474/954433688.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Normalized Score'] = np.random.uniform(0.1, 1.0, len(df))


Initializing base location data...
Initializing time engine...


Generating RNN-Ready Dataset: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 50000/50000 [01:03<00:00, 790.40it/s]



Dataset generated successfully: rnn_ready_complaints_v6.csv


,ticket_id,timestamp,location,pincode,severity_rank,grievance_text
26504,TKT026505,2023-01-01 00:59:00,Nehrunagar,500026,2,wifi is not working again.
49268,TKT049269,2023-01-01 01:16:00,Ramkoti,500095,4,Watching videos is buffering sometimes.
16393,TKT016394,2023-01-01 01:21:00,Khapra,500062,4,Watching videos is buffering sometimes.
34853,TKT034854,2023-01-01 01:49:00,Kyasaram,500062,1,URGENT: No signal. Work affected. Fix immediat...
18714,TKT018715,2023-01-01 02:11:00,Moosapet,500018,2,Facing total connection failure since morning.


In [ ]:
# Grouping by severity rank and finding unique values for key columns
unique_vals_per_grp = rnn_ready_df.groupby('severity_rank').agg({
    'grievance_text': 'unique',
    'location': 'nunique',
    'ticket_id': 'count'
}).rename(columns={'ticket_id': 'total_records'})

print("Summary of unique values per Severity Rank (1-5):")
display(unique_vals_per_grp)

# Printing a few unique grievance examples for each rank to verify linguistic separation
print("\n--- Sample Grievances per Rank ---")
for rank in sorted(rnn_ready_df['severity_rank'].unique()):
    samples = rnn_ready_df[rnn_ready_df['severity_rank'] == rank]['grievance_text'].unique()[:3]
    print(f"Rank {rank}: {list(samples)}")

Summary of unique values per Severity Rank (1-5):


,grievance_text,location,total_records
severity_rank,,,
1,[URGENT: No signal. Work affected. Fix immedia...,228,7617
2,"[wifi is not working again., Facing total conn...",228,12329
3,"[Internet is intermittent and unstable., inter...",228,14956
4,"[Watching videos is buffering sometimes., Ping...",228,9991
5,"[Is there any maintenance today? Thank you., I...",228,5107



--- Sample Grievances per Rank ---
Rank 1: ['URGENT: No signal. Work affected. Fix immediately.', 'TOTAL BLACKOUT. wifi NOT WORKING. FIX ASAP!!', 'connection down since 2 days. This is unacceptable!!']
Rank 2: ['wifi is not working again.', 'Facing total connection failure since morning.', 'Why is my wifi down? Please resolve.']
Rank 3: ['Internet is intermittent and unstable.', 'internet keeps dropping every 10 minutes.', 'Very slow speeds for the last 5 hours.']
Rank 4: ['Watching videos is buffering sometimes.', 'Ping is high in gaming, check please.', 'Speed is a bit slow today.']
Rank 5: ['Is there any maintenance today? Thank you.', 'I want to check my data balance.', 'Please tell me about new plans.']


In [ ]:
# Calculating unique values for each column in the RNN-Ready V6 dataset
unique_counts_v6 = rnn_ready_df.nunique()

print("Unique values in each column (V6 - Total Records: 50,000):")
print(unique_counts_v6)

# Also displaying value counts for the severity_rank to see the distribution
print("\nDistribution of Severity Ranks:")
print(rnn_ready_df['severity_rank'].value_counts().sort_index())

Unique values in each column (V6 - Total Records: 50,000):
ticket_id         50000
timestamp         48857
location            228
pincode              67
severity_rank         5
grievance_text       38
dtype: int64

Distribution of Severity Ranks:
severity_rank
1     7617
2    12329
3    14956
4     9991
5     5107
Name: count, dtype: int64


In [ ]:
df = df.dropna(subset=["Pincode", "Location"]).reset_index(drop=True)
df["Pincode"] = df["Pincode"].astype(int)

In [ ]:
!pip install faker tqdm -q

import pandas as pd
import numpy as np
import random
from faker import Faker
from tqdm import tqdm
import uuid
from datetime import timedelta

# -----------------------------
# CONFIG
# -----------------------------
N = 50000
fake = Faker('en_IN')

# -----------------------------
# BASE DATAFRAME CHECK
# -----------------------------
if 'df' not in globals():
    print("âš ï¸ Base dataframe 'df' not found. Please run your first cell.")
else:
    if 'Normalized Score' not in df.columns:
        df['Normalized Score'] = np.random.uniform(0.1, 1.0, len(df))

    # -----------------------------
    # ISP Complaint Types
    # -----------------------------
    complaint_types = {
        "No Internet": 0.30,
        "Slow Speed": 0.20,
        "Intermittent": 0.15,
        "High Latency": 0.10,
        "Packet Loss": 0.08,
        "Router Issue": 0.10,
        "DNS Issue": 0.07
    }

    # -----------------------------
    # NEW: Severity Mapping (5 classes)
    # -----------------------------
    severity_map = {
    "No Internet": ["High", "Critical"],
    "Slow Speed": ["Very Low", "Low", "Medium"],
    "Intermittent": ["Medium", "High"],
    "High Latency": ["Medium", "High"],
    "Packet Loss": ["Medium", "High"],
    "Router Issue": ["Very Low", "Low", "Medium"],
    "DNS Issue": ["Very Low", "Low", "Medium"]
}

    severity_weights = {
        "Very Low": 0.10,
        "Low": 0.25,
        "Medium": 0.30,
        "High": 0.20,
        "Critical": 0.15
    }

    # -----------------------------
    # ADDRESS
    # -----------------------------
    def generate_full_address(location, pincode):
        base_address = fake.address().split('\n')[0]
        return f"{base_address}, {location}, Hyderabad, {int(pincode)}"

    # -----------------------------
    # TEXT GENERATOR (IMPROVED)
    # -----------------------------
    def fallback_text(ctype, tone, lang):
        base_phrases = {
            "No Internet": ["internet not working", "wifi down", "no connection", "no internet access"],
            "Slow Speed": ["internet very slow", "speed is low", "buffering issue", "net slow"],
            "Intermittent": ["disconnecting again and again", "unstable connection", "keeps dropping"],
            "High Latency": ["ping too high", "lag issue", "delay in response"],
            "Packet Loss": ["packet loss happening", "data loss issue"],
            "Router Issue": ["router not working", "router keeps restarting"],
            "DNS Issue": ["dns not resolving", "cannot access websites"]
        }

        patterns = [
            "{issue}",
            "{issue} since morning",
            "facing {issue}",
            "{issue} please fix",
            "still {issue}",
            "{issue} from yesterday"
        ]

        issue = random.choice(base_phrases[ctype])
        sentence = random.choice(patterns).format(issue=issue)

        if tone == "angry":
            sentence = sentence.upper() + "!!!"
        elif tone == "polite":
            sentence = "please " + sentence

        if lang == "hinglish":
            sentence += " yaar"

        return sentence

    # -----------------------------
    # LANGUAGE STYLE
    # -----------------------------
    def get_language_style(area_category):
        if "IT" in str(area_category):
            return "english"
        return random.choice(["english", "hinglish"])

    # -----------------------------
    # CUSTOMER GENERATION
    # -----------------------------
    customer_pool = []

    def get_customer(loc, pin):
        if random.random() < 0.15 and customer_pool:
            return random.choice(customer_pool)
        else:
            cust_id = f"CUST-{random.randint(100000, 999999)}"
            address = generate_full_address(loc, pin)
            customer_pool.append((cust_id, address))
            return cust_id, address

    # -----------------------------
    # TIME ENGINE (2 YEARS)
    # -----------------------------
    dates = pd.date_range(start="2023-01-01", end="2024-12-31", freq="H")
    time_engine = pd.DataFrame({"timestamp": dates})

    # -----------------------------
    # GENERATE DATA
    # -----------------------------
    data = []
    area_weights = df['Normalized Score'].fillna(0.0001)

    for i in tqdm(range(N), desc="Generating ISP Complaints (5-Class)"):

        loc_row = df.sample(n=1, weights=area_weights).iloc[0]
        time_row = time_engine.sample(n=1).iloc[0]

        ts = time_row['timestamp'] + timedelta(minutes=random.randint(0, 59))

        # Complaint Type
        ctype = np.random.choice(
            list(complaint_types.keys()),
            p=list(complaint_types.values())
        )

        # Severity assignment
        if ctype in severity_map:
            severity = random.choice(severity_map[ctype])
        else:
            severity = np.random.choice(
                list(severity_weights.keys()),
                p=list(severity_weights.values())
            )

        lang = get_language_style(loc_row.get('Category', 'Residential'))
        tone = random.choice(["normal", "angry", "polite"])

        text = fallback_text(ctype, tone, lang)

        cust_id, address = get_customer(loc_row['Location'], loc_row['Pincode'])

        data.append({
            "ticket_id": f"TKT{i+1:06d}",
            "customer_id": cust_id,
            "timestamp": ts,
            "location": loc_row['Location'],
            "pincode": int(loc_row['Pincode']),
            "complaint_type": ctype,
            "severity": severity,   # ðŸŽ¯ TARGET COLUMN
            "grievance_text": text,
            "customer_address": address
        })

    # -----------------------------
    # FINAL DATAFRAME
    # -----------------------------
    final_df = pd.DataFrame(data).sort_values("timestamp")

    final_df.to_csv("data/raw/isp_complaints_5class_50k.csv", index=False)

    print("âœ… Dataset Generated Successfully!")
    display(final_df.head())

    print("\nðŸ“Š Severity Distribution:\n")
    print(final_df["severity"].value_counts())

    print("\nðŸ“Š Complaint Type Distribution:\n")
    print(final_df["complaint_type"].value_counts())

/tmp/ipykernel_3268/2094815300.py:128: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  dates = pd.date_range(start="2023-01-01", end="2024-12-31", freq="H")
Generating ISP Complaints (5-Class): 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 50000/50000 [01:12<00:00, 693.29it/s]


âœ… Dataset Generated Successfully!


,ticket_id,customer_id,timestamp,location,pincode,complaint_type,severity,grievance_text,customer_address
16401,TKT016402,CUST-651110,2023-01-01 00:06:00,Reinbazar,500023,Intermittent,High,please still keeps dropping,"57, Mitter Nagar, Reinbazar, Hyderabad, 500023"
11522,TKT011523,CUST-949032,2023-01-01 00:10:00,Anandnagar,500004,No Internet,Critical,please facing no internet access yaar,"71, Anandnagar, Hyderabad, 500004"
22438,TKT022439,CUST-569773,2023-01-01 00:17:00,Kattedan Ie,500077,Router Issue,Low,please still router not working,"09/91, Kattedan Ie, Hyderabad, 500077"
36678,TKT036679,CUST-631193,2023-01-01 00:48:00,Moghalpura,500002,Packet Loss,Medium,facing data loss issue yaar,"H.No. 329, Acharya Zila, Hubliâ€“Dharwad-094212,..."
17065,TKT017066,CUST-449991,2023-01-01 01:16:00,Shyam Nagar,500004,Slow Speed,Very Low,INTERNET VERY SLOW SINCE MORNING!!! yaar,"064, Mukherjee Marg, Kurnool-580143, Shyam Nag..."



ðŸ“Š Severity Distribution:

severity
High        15874
Medium      14409
Critical     7369
Very Low     6215
Low          6133
Name: count, dtype: int64

ðŸ“Š Complaint Type Distribution:

complaint_type
No Internet     14908
Slow Speed      10041
Intermittent     7507
High Latency     5044
Router Issue     4950
Packet Loss      3932
DNS Issue        3618
Name: count, dtype: int64


In [ ]:
# Load the newly generated 5-class dataset
isp_5class_df = pd.read_csv('data/raw/isp_complaints_5class_50k.csv')

# Calculate unique counts for all columns
unique_summary = isp_5class_df.nunique()

print("Unique Values in 'data/raw/isp_complaints_5class_50k.csv' (Total Records: 50,000):")
print(unique_summary)

# Breakdown of unique grievance texts per Severity class
print("\nUnique Grievance Texts per Severity Level:")
grp_unique = isp_5class_df.groupby('severity')['grievance_text'].nunique()
print(grp_unique)

# Display a few samples for each severity to verify distinction
print("\n--- Sample Grievances by Severity ---")
for sev in isp_5class_df['severity'].unique():
    sample = isp_5class_df[isp_5class_df['severity'] == sev]['grievance_text'].unique()[:2]
    print(f"[{sev}]: {list(sample)}")

Unique Values in 'data/raw/isp_complaints_5class_50k.csv' (Total Records: 50,000):
ticket_id           50000
customer_id         41419
timestamp           48885
location              228
pincode                67
complaint_type          7
severity                5
grievance_text        720
customer_address    41689
dtype: int64

Unique Grievance Texts per Severity Level:
severity
Critical    144
High        432
Low         288
Medium      576
Very Low    288
Name: grievance_text, dtype: int64

--- Sample Grievances by Severity ---
[High]: ['please still keeps dropping', 'facing packet loss happening']
[Critical]: ['please facing no internet access yaar', 'STILL INTERNET NOT WORKING!!! yaar']
[Low]: ['please still router not working', 'please router keeps restarting from yesterday']
[Medium]: ['facing data loss issue yaar', 'FACING DELAY IN RESPONSE!!! yaar']
[Very Low]: ['INTERNET VERY SLOW SINCE MORNING!!! yaar', 'STILL INTERNET VERY SLOW!!!']
